# Stock Market Prediction: A Comparative Analysis
**Objective**: To forecast the closing price of Bajaj Finance using Deep Learning (LSTM) and benchmark its performance against traditional time-series and machine learning models (ARIMA, GARCH, ETS, XGBoost).

In [ ]:
!pip install arch xgboost

## 1. Import Libraries and Load Data
We begin by importing the necessary libraries for data manipulation, time-series forecasting, and evaluation. Warnings are suppressed to maintain a clean output for the analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import statsmodels.api as sm
from arch import arch_model
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import xgboost as xgb

import warnings
warnings.filterwarnings('ignore')

## 2. Data Preprocessing
We load the historical stock data, format the date index, and scale our features to prepare them for the neural network. We will use a multi-feature approach (`Close`, `Open`, `High`, `Low`, `Volume`) with a sequential look-back window of 120 days.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# 1. Dynamically find all CSV files containing 'BAJFINANCE' in the name
csv_files = glob.glob('/content/*BAJFINANCE*.csv')

if not csv_files:
    raise FileNotFoundError("No BAJFINANCE CSV file found in /content/. Please ensure it is uploaded.")

# 2. Select the most recently modified file to avoid conflicts with older duplicates
latest_csv = max(csv_files, key=os.path.getmtime)
print(f"Auto-loaded dataset: {latest_csv}")

# Load and preprocess the data
df = pd.read_csv(latest_csv)

# Ensure the Date column is in the correct format and set as index
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

# Drop NaN values
df = df.dropna()

# Scale the features 'Close', 'Open', 'High', 'Low', and 'Volume'
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df[['Close', 'Open', 'High', 'Low', 'Volume']])

# Create datasets with multiple features
def create_dataset_multifeature(dataset, look_back=120):
    X, y = [], []
    for i in range(look_back, len(dataset)):
        X.append(dataset[i-look_back:i])  # Include all features
        y.append(dataset[i, 0])           # Predict 'Close' price (index 0)
    return np.array(X), np.array(y)

# Increase the look-back window to capture longer trends
look_back = 120
X, y = create_dataset_multifeature(scaled_data, look_back)

# Reshape X to be [samples, time steps, features]
X = np.reshape(X, (X.shape[0], X.shape[1], X.shape[2]))

# Split the data into training and testing sets (Sequential split for time-series)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

## 3. LSTM Model Construction
We design a multi-layer Long Short-Term Memory (LSTM) network with dropout layers to prevent overfitting. Early stopping is implemented to halt training when the validation loss plateaus.

In [ ]:
# Build the LSTM model with increased complexity
model = Sequential()
model.add(LSTM(units=100, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dropout(0.3))  
model.add(LSTM(units=100, return_sequences=False))
model.add(Dropout(0.3))
model.add(Dense(units=50)) 
model.add(Dense(units=1))  

# Compile the model with a lower learning rate
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

# Train the model with early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(X_train, y_train, batch_size=32, epochs=100, validation_split=0.2, callbacks=[early_stopping], verbose=1)

# Generate Predictions
predictions = model.predict(X_test)

# Inverse transformation for predictions (restoring original scale)
predictions = scaler.inverse_transform(
    np.concatenate((predictions, np.zeros((predictions.shape[0], X_test.shape[2]-1))), axis=1)
)[:, 0]

# Inverse transform y_test for accurate metric comparison
y_test_scaled = scaler.inverse_transform(
    np.concatenate((y_test.reshape(-1, 1), np.zeros((y_test.shape[0], X_test.shape[2]-1))), axis=1)
)[:, 0]

## 4. Traditional & Machine Learning Baselines
To validate the robustness of the LSTM model, we benchmark it against traditional statistical methods (ARIMA, GARCH, ETS) and a tree-based ensemble method (XGBoost).

In [ ]:
# --- 4.1 ARIMA Model ---
arima_order = (5, 1, 0)  # (p, d, q)
arima_model = sm.tsa.ARIMA(df['Close'][:len(X_train) + look_back], order=arima_order)
arima_fitted = arima_model.fit()
arima_predictions = arima_fitted.forecast(steps=len(X_test))

# Convert and inverse scale
arima_predictions_np = np.array(arima_predictions).reshape(-1, 1)
arima_predictions_scaled = scaler.inverse_transform(
    np.concatenate((arima_predictions_np, np.zeros((arima_predictions_np.shape[0], X_test.shape[2]-1))), axis=1)
)[:, 0]

# --- 4.2 GARCH Model ---
garch_model = arch_model(df['Close'][:len(X_train) + look_back], vol='Garch', p=1, q=1)
garch_fitted = garch_model.fit(disp="off")
garch_forecast = garch_fitted.forecast(horizon=len(X_test))
garch_predictions = garch_forecast.mean.values[-1, :]

# Convert and inverse scale
garch_predictions_scaled = scaler.inverse_transform(
    np.concatenate((garch_predictions.reshape(-1, 1), np.zeros((garch_predictions.shape[0], X_test.shape[2]-1))), axis=1)
)[:, 0]

# --- 4.3 Exponential Smoothing (ETS) Model ---
ets_model = ExponentialSmoothing(df['Close'][:len(X_train) + look_back], trend='add', seasonal='add', seasonal_periods=12)
ets_fitted = ets_model.fit()
ets_predictions = ets_fitted.forecast(steps=len(X_test))

# Convert and inverse scale
ets_predictions_np = np.array(ets_predictions).reshape(-1, 1)
ets_predictions_scaled = scaler.inverse_transform(
    np.concatenate((ets_predictions_np, np.zeros((ets_predictions_np.shape[0], X_test.shape[2]-1))), axis=1)
)[:, 0]

# --- 4.4 XGBoost Model ---
xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1)
xgb_model.fit(X_train.reshape(X_train.shape[0], -1), y_train)
xgb_predictions = xgb_model.predict(X_test.reshape(X_test.shape[0], -1))

# Convert and inverse scale
xgb_predictions_scaled = scaler.inverse_transform(
    np.concatenate((xgb_predictions.reshape(-1, 1), np.zeros((xgb_predictions.shape[0], X_test.shape[2]-1))), axis=1)
)[:, 0]

## 5. Evaluation and Visualization
Finally, we evaluate all models utilizing Mean Squared Error (MSE) and Mean Absolute Error (MAE), visualizing their prediction trajectories directly against the actual historical stock prices.

In [ ]:
# Print Evaluation Metrics
print("--- Model Performance Metrics ---")
models = {
    'ARIMA': arima_predictions_scaled, 
    'GARCH': garch_predictions_scaled, 
    'ETS': ets_predictions_scaled, 
    'XGBoost': xgb_predictions_scaled,
    'LSTM': predictions
}

for model_name, model_preds in models.items():
    mse = mean_squared_error(y_test_scaled, model_preds)
    mae = mean_absolute_error(y_test_scaled, model_preds)
    print(f"{model_name:<8} - MSE: {mse:<20.2f} MAE: {mae:.2f}")

# Plotting the results
plt.figure(figsize=(16, 20))
test_dates = df.index[-len(y_test):]

plot_configs = [
    (1, arima_predictions_scaled, 'ARIMA', 'red'),
    (2, garch_predictions_scaled, 'GARCH', 'blue'),
    (3, ets_predictions_scaled, 'ETS', 'purple'),
    (4, xgb_predictions_scaled, 'XGBoost', 'brown')
]

for subplot_idx, preds, name, color in plot_configs:
    plt.subplot(5, 1, subplot_idx)
    plt.plot(test_dates, y_test_scaled, label='Actual Price', color='orange')
    plt.plot(test_dates, predictions, label='LSTM Predictions', color='green', linewidth=2)
    plt.plot(test_dates, preds, label=f'{name} Predictions', color=color)
    plt.title(f'{name} vs LSTM')
    plt.legend()

# All models combined
plt.subplot(5, 1, 5)
plt.plot(test_dates, y_test_scaled, label='Actual Price', color='orange', linewidth=2)
plt.plot(test_dates, predictions, label='LSTM', color='green', linewidth=2)
plt.plot(test_dates, arima_predictions_scaled, label='ARIMA', color='red', alpha=0.6)
plt.plot(test_dates, garch_predictions_scaled, label='GARCH', color='blue', alpha=0.6)
plt.plot(test_dates, ets_predictions_scaled, label='ETS', color='purple', alpha=0.6)
plt.plot(test_dates, xgb_predictions_scaled, label='XGBoost', color='brown', alpha=0.6)
plt.title('All Models vs LSTM')
plt.legend()

plt.tight_layout()
plt.show()